# Update log
2024/08/29
- Test without Responsivity and/or baseline
- Test with reduced number of features
- Baseline alone gives up to 75% accuracy
- Suspect data distribution due to always running experiment in 0,1,2,3,4
- Run experiment in De Bruijn sequence to balance the adjacent experiment channels
---

In [263]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import json

spk_data = "D:\\code\\uom_explore\\model_input\\3_features.csv"
spk_pca_data = "D:\\code\\uom_explore\\model_input\\pca_df.csv"
spk_20 = "D:\\code\\uom_explore\\data_science\\reduced\\20_140_195_df.csv"
spk_full = "D:\\code\\uom_explore\\data_science\\df.csv"
debruijn_1 = "D:\\code\\uom_explore\\processed_data\\metrics_exp_brujin_seq_1.csv"

hkr_wsl_data = "/home/hk-wsl/code/uom_explore/model_input/feature_matrix.csv"
hkr_pca_data = "/home/hk-wsl/code/uom_explore/model_input/feature_pca.csv"

spk_json = "/home/gavinlouuu/coding/uom_explore/data_science/parameter.json"
hkr_wsl_json = "/home/hk-wsl/code/uom_explore/data_science/parameter.json"

data_path = debruijn_1
param_path = spk_json

with open('parameter.json','r') as file:
    params = json.load(file)

# Hyperparameters
# Extract parameters from the JSON object
hidden_size = params['mlp']['hidden_size']
ground_truth = params['ground_truth']
num_epochs = params['mlp']['num_epochs']
batch_size = params['mlp']['batch_size']
learning_rate = params['mlp']['learning_rate']
momentum_value = params['mlp']['momentum_value']
dropout_rate = params['mlp']['dropout']
weight_decay = params['mlp']['weight_decay']

scheduler_params = params['mlp']['scheduler']

df = pd.read_csv(data_path)
# drop experiment_id column
# df.drop('experiment_id', axis=1, inplace=True)
print(type(df))
df.head()



<class 'pandas.core.frame.DataFrame'>


,experiment_id,channel_id,baseline_160,baseline_162,baseline_165,baseline_167,baseline_170,baseline_172,baseline_175,baseline_177,...,temperature_max,temperature_std,humidity_mean,humidity_min,humidity_max,humidity_std,pressure_mean,pressure_min,pressure_max,pressure_std
0,20240829145200s1c0r0,0,15333.197116,22957.351691,23848.697631,23071.544934,21668.667869,20477.232045,19239.758925,18185.434345,...,33.21,0.333116,64.992000,60.89,68.48,2.459250,100210.552941,100201.0,100213.0,1.802814
1,20240829145235s1c1r0,1,21815.038599,33226.561073,34565.623032,33401.419655,31172.487756,29492.245969,27552.348528,25827.395031,...,33.29,0.036399,62.682083,60.15,65.88,1.926198,100210.319444,100209.0,100212.0,0.667693
2,20240829145311s1c2r0,2,28245.293614,44118.401113,45736.157153,43995.030951,41154.533173,38596.182230,35870.663615,33576.358547,...,33.40,0.039994,59.122069,58.56,59.73,0.344570,100210.344828,100208.0,100212.0,0.899969
3,20240829145346s1c3r0,3,32950.229651,52038.602759,54191.269085,51752.884670,47946.761953,45108.761385,42011.398789,39147.586067,...,33.44,0.029688,59.461351,58.47,60.65,0.728982,100207.351351,100204.0,100209.0,1.127884
4,20240829145422s1c4r0,4,18762.637612,28167.472573,28372.246360,26803.690386,24567.917448,22757.383722,20645.451620,19187.641379,...,33.49,0.029308,60.926180,58.38,64.68,2.028016,100207.573034,100205.0,100210.0,1.185975


## Load all features

In [264]:
# Get all column names from the DataFrame
all_columns = df.columns.tolist()

# Remove 'channel_id' and the ground truth from the list of features
features = [col for col in all_columns if col != 'experiment_id' and col != ground_truth]

# X includes all features
X = df[features]

# Print the features
print("Features:")
print(json.dumps(features, indent=2))



Features:
[
  "baseline_160",
  "baseline_162",
  "baseline_165",
  "baseline_167",
  "baseline_170",
  "baseline_172",
  "baseline_175",
  "baseline_177",
  "baseline_180",
  "baseline_182",
  "baseline_185",
  "baseline_187",
  "baseline_190",
  "baseline_192",
  "baseline_195",
  "baseline_197",
  "baseline_200",
  "baseline_202",
  "baseline_205",
  "baseline_210",
  "baseline_212",
  "baseline_215",
  "baseline_217",
  "baseline_220",
  "baseline_222",
  "baseline_225",
  "baseline_227",
  "baseline_230",
  "baseline_232",
  "baseline_235",
  "baseline_237",
  "max_reaction_R_160",
  "max_reaction_R_162",
  "max_reaction_R_165",
  "max_reaction_R_167",
  "max_reaction_R_170",
  "max_reaction_R_172",
  "max_reaction_R_175",
  "max_reaction_R_177",
  "max_reaction_R_180",
  "max_reaction_R_182",
  "max_reaction_R_185",
  "max_reaction_R_187",
  "max_reaction_R_190",
  "max_reaction_R_192",
  "max_reaction_R_195",
  "max_reaction_R_197",
  "max_reaction_R_200",
  "max_reaction_R_202"

# Adjust features

## Select settings to keep

In [265]:
# # Preserve BME features 
# bme_features = [col for col in features if any(suffix in col for suffix in ['_min','_max','_mean','_std'])]

# # Extract all unique numbers from feature names
# feature_numbers = set()
# for feature in features:
#     parts = feature.split('_')
#     if len(parts) > 1 and parts[-1].isdigit():
#         feature_numbers.add(int(parts[-1]))

# # Sort the numbers
# sorted_numbers = sorted(feature_numbers)

# # Select settings to keep
# settings_to_keep = []

# # Filter the features to keep only the selected settings
# features_to_keep = [feature for feature in features if feature.split('_')[-1].isdigit() and int(feature.split('_')[-1]) in settings_to_keep]

# # Add back the BME features
# features_to_keep.extend(bme_features)

# # Update the features list
# features = features_to_keep

# # Print the features
# print("Features:")
# print(json.dumps(features, indent=2))

## Select features to remove

In [266]:
# Remove features with _min, _max, and _std suffixes
features_to_keep = [col for col in features if not any(suffix in col for suffix in ['_std'])]

# Update the features list
features = features_to_keep

# Update X dataframe to only include the kept features
X = df[features]

# Update input_size
input_size = len(features)

# print(f"Features after removing '_min', '_max' and '_std' suffixes:")
print(json.dumps(features, indent=2))
# print(f"New input size: {input_size}")



[
  "baseline_160",
  "baseline_162",
  "baseline_165",
  "baseline_167",
  "baseline_170",
  "baseline_172",
  "baseline_175",
  "baseline_177",
  "baseline_180",
  "baseline_182",
  "baseline_185",
  "baseline_187",
  "baseline_190",
  "baseline_192",
  "baseline_195",
  "baseline_197",
  "baseline_200",
  "baseline_202",
  "baseline_205",
  "baseline_210",
  "baseline_212",
  "baseline_215",
  "baseline_217",
  "baseline_220",
  "baseline_222",
  "baseline_225",
  "baseline_227",
  "baseline_230",
  "baseline_232",
  "baseline_235",
  "baseline_237",
  "max_reaction_R_160",
  "max_reaction_R_162",
  "max_reaction_R_165",
  "max_reaction_R_167",
  "max_reaction_R_170",
  "max_reaction_R_172",
  "max_reaction_R_175",
  "max_reaction_R_177",
  "max_reaction_R_180",
  "max_reaction_R_182",
  "max_reaction_R_185",
  "max_reaction_R_187",
  "max_reaction_R_190",
  "max_reaction_R_192",
  "max_reaction_R_195",
  "max_reaction_R_197",
  "max_reaction_R_200",
  "max_reaction_R_202",
  "max_r

# Data split and scale

In [267]:
# Preview features
X = X[features]
# print(X.head())


In [268]:
input_size = len(X.columns)  # removing the ground truth from the number of columns counted
num_classes = df[ground_truth].nunique()
print(f"Number of classes: {num_classes}")
print(f"Number of features: {input_size}")

# Preview ground truth
y = df[ground_truth]


# Split into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)  # This makes 60%, 20%, 20%

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler to the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to validation and test sets
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert arrays to NumPy arrays (no need to convert to PyTorch tensors)
X_train_np = X_train_scaled
y_train_np = y_train.to_numpy()
X_val_np = X_val_scaled
y_val_np = y_val.to_numpy()
X_test_np = X_test_scaled
y_test_np = y_test.to_numpy()


Number of classes: 5
Number of features: 102


# Random Forest

In [269]:
n_estimators = params['rf']['n_estimators']
random_state = params['rf']['random_state']
# Initialize the model
rf_model = RandomForestClassifier(
    n_estimators=n_estimators,
    random_state=random_state
)


# Function to predict using the Random Forest model
def predict_rf(model, data):
    return model.predict(data)



# Function to train and evaluate the Random Forest model
def train_and_evaluate_rf(model, X_train, y_train, X_val, y_val):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions on training and validation sets
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # Calculate accuracies
    train_accuracy = accuracy_score(y_train, y_train_pred)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    
    print(f'Train Accuracy: {train_accuracy:.4f}, Validation Accuracy: {val_accuracy:.4f}')
    
    return model, train_accuracy, val_accuracy

# Train and evaluate the Random Forest model
rf_model, rf_train_accuracy, rf_val_accuracy = train_and_evaluate_rf(
    rf_model, 
    X_train_scaled, y_train, 
    X_val_scaled, y_val
)

# Evaluate on the test set
y_test_pred = predict_rf(rf_model, X_test_scaled)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')


Train Accuracy: 1.0000, Validation Accuracy: 0.7750
Test Accuracy: 0.8500


# Gradient Boost

In [270]:
# Import necessary libraries
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Define parameters for Gradient Boosting
n_estimators = params['gb']['n_estimators']
learning_rate = params['gb']['learning_rate']
max_depth = params['gb']['max_depth']
random_state = params['gb']['random_state']

# Initialize the Gradient Boosting model
gb_model = GradientBoostingClassifier(
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=max_depth,
    random_state=random_state
)

# Function to predict using the Gradient Boosting model
def predict_gb(model, data):
    return model.predict(data)

# Function to train and evaluate the Gradient Boosting model
def train_and_evaluate_gb(model, X_train, y_train, X_val, y_val):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions on training and validation sets
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # Calculate accuracies
    train_accuracy = accuracy_score(y_train, y_train_pred)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    
    print(f'Train Accuracy: {train_accuracy:.4f}, Validation Accuracy: {val_accuracy:.4f}')
    
    return model, train_accuracy, val_accuracy

# Train and evaluate the Gradient Boosting model
gb_model, gb_train_accuracy, gb_val_accuracy = train_and_evaluate_gb(
    gb_model, 
    X_train_scaled, y_train, 
    X_val_scaled, y_val
)

# Evaluate on the test set
y_test_pred = predict_gb(gb_model, X_test_scaled)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')

Train Accuracy: 1.0000, Validation Accuracy: 0.7000
Test Accuracy: 0.7500


# Stacking Ensemble

In [271]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

# Split the data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Define the base models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

# Define the meta-model with increased max_iter and scaling
meta_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# Create the stacking ensemble
stacking_ensemble = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

# Train the stacking ensemble
stacking_ensemble.fit(X_train, y_train)

# Evaluate the model on the validation set
y_val_pred = stacking_ensemble.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f'Validation Accuracy: {val_accuracy:.4f}')

# Cross-validation scores
cv_scores = cross_val_score(stacking_ensemble, X_train, y_train, cv=5)
print(f'Cross-Validation Scores: {cv_scores}')
print(f'Mean CV Score: {np.mean(cv_scores):.4f}')

# Confusion Matrix for validation set
conf_matrix_val = confusion_matrix(y_val, y_val_pred)
print('Confusion Matrix (Validation):')
print(conf_matrix_val)

# Classification Report for validation set
class_report_val = classification_report(y_val, y_val_pred)
print('Classification Report (Validation):')
print(class_report_val)

# Evaluate the model on the test set
y_test_pred = stacking_ensemble.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')

# Confusion Matrix for test set
conf_matrix_test = confusion_matrix(y_test, y_test_pred)
print('Confusion Matrix (Test):')
print(conf_matrix_test)

# Classification Report for test set
class_report_test = classification_report(y_test, y_test_pred)
print('Classification Report (Test):')
print(class_report_test)

# Feature Importances from base models
for name, model in base_models:
    if hasattr(model, 'feature_importances_'):
        print(f'Feature importances for {name}:')
        print(model.feature_importances_)

Validation Accuracy: 0.8667
Cross-Validation Scores: [0.71875 0.75    0.78125 0.84375 0.78125]
Mean CV Score: 0.7750
Confusion Matrix (Validation):
[[9 4 0 0 0]
 [0 7 0 0 0]
 [0 0 3 0 0]
 [0 0 0 4 0]
 [0 0 0 0 3]]
Classification Report (Validation):
              precision    recall  f1-score   support

           0       1.00      0.69      0.82        13
           1       0.64      1.00      0.78         7
           2       1.00      1.00      1.00         3
           3       1.00      1.00      1.00         4
           4       1.00      1.00      1.00         3

    accuracy                           0.87        30
   macro avg       0.93      0.94      0.92        30
weighted avg       0.92      0.87      0.87        30

Test Accuracy: 0.9000
Confusion Matrix (Test):
[[0 1 0 0 0]
 [0 3 0 0 0]
 [0 0 1 0 0]
 [0 0 0 3 0]
 [0 0 0 0 2]]
Classification Report (Test):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
         

d:\code\uom_explore\.env\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\code\uom_explore\.env\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\code\uom_explore\.env\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Voting Classifier

In [272]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

# First, split the data into train+val and test sets
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Then split the train+val set into separate train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)  # 0.25 x 0.8 = 0.2

# Define the base models with scaling for Logistic Regression
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('lr', make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))
]

# Create the voting ensemble
voting_ensemble = VotingClassifier(estimators=base_models, voting='hard')  # 'soft' for probability averaging

# Train the voting ensemble
voting_ensemble.fit(X_train, y_train)

# Evaluate on validation set
y_val_pred = voting_ensemble.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f'Validation Accuracy: {val_accuracy:.4f}')

# Cross-validation scores
cv_scores = cross_val_score(voting_ensemble, X_train, y_train, cv=5)
print(f'Cross-Validation Scores: {cv_scores}')
print(f'Mean CV Score: {np.mean(cv_scores):.4f}')

# Confusion Matrix for validation set
conf_matrix_val = confusion_matrix(y_val, y_val_pred)
print('Validation Confusion Matrix:')
print(conf_matrix_val)

# Classification Report for validation set
class_report_val = classification_report(y_val, y_val_pred)
print('Validation Classification Report:')
print(class_report_val)

# Final evaluation on test set
y_test_pred = voting_ensemble.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'\nTest Accuracy: {test_accuracy:.4f}')

# Confusion Matrix for test set
conf_matrix_test = confusion_matrix(y_test, y_test_pred)
print('Test Confusion Matrix:')
print(conf_matrix_test)

# Classification Report for test set
class_report_test = classification_report(y_test, y_test_pred)
print('Test Classification Report:')
print(class_report_test)

Validation Accuracy: 0.7750
Cross-Validation Scores: [0.79166667 0.75       0.83333333 0.83333333 0.875     ]
Mean CV Score: 0.8167
Validation Confusion Matrix:
[[ 4  5  0  0  0]
 [ 2  4  0  0  0]
 [ 0  0  5  2  0]
 [ 0  0  0 10  0]
 [ 0  0  0  0  8]]
Validation Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.44      0.53         9
           1       0.44      0.67      0.53         6
           2       1.00      0.71      0.83         7
           3       0.83      1.00      0.91        10
           4       1.00      1.00      1.00         8

    accuracy                           0.78        40
   macro avg       0.79      0.77      0.76        40
weighted avg       0.80      0.78      0.77        40


Test Accuracy: 0.8000
Test Confusion Matrix:
[[7 7 0 0 0]
 [1 9 0 0 0]
 [0 0 4 0 0]
 [0 0 0 7 0]
 [0 0 0 0 5]]
Test Classification Report:
              precision    recall  f1-score   support

           0       0.88      0